# B3 — did attempt 6 actually fix the collapse, or just hide it from pooled RMS?

**Why this notebook exists**: attempt 6's `region_avg_sigma_xx_direct` pooled RMS (84.11%) is the smallest seen across all 6 mixed-stress attempts -- but it is also BELOW attempt 4's own 96.1%, which this project already established is itself the signature of a trivial near-zero predictor (pooled RMS relative error -> ~100% when the prediction is ~0, since error ~= true value). The `constitutive` training loss also plateaued at 0.80-0.89, nearly identical to attempt 4's own already-diagnosed collapse range. Correlation is almost unchanged from attempt 5 (0.6533 vs 0.6531), while `sign_agree` jumped from 0.17 to 0.83 -- genuinely ambiguous from aggregate stats alone.

**What this notebook does**: loads attempt 6's EXISTING checkpoint (no training, no new GPU time beyond one forward pass) and directly reports predicted-P's own mean-abs/std against the true P field's own stats, computed fresh from the held-out FEM set -- the exact same statistic that originally confirmed attempt 4's collapse (predicted mean abs ~0.73-1.27 vs. true ~68). Should finish in well under a minute.

In [ ]:
# =====================================================================
#  CELL -- cheap, no-training diagnostic: is attempt 6's predicted P
#  collapsed (near-constant, near-zero) or genuinely scaled/varied?
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys

def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU is fine -- one forward pass)')

R = '/content/drive/MyDrive/pfem_run'
CHECKPOINT = f'{R}/b3_mixed_stress_local_refine_relative/checkpoint_50000.pt'
DATASET = f'{R}/b3_dataset_clean_holdout/dataset.h5'

sys.argv = [
    'inspect_mixed_stress_P_scale.py',
    '--checkpoint', CHECKPOINT,
    '--dataset', DATASET,
    '--n_samples', '16',
]
from omar_pfem.inspect_mixed_stress_P_scale import main as inspect_main
inspect_main()

print('\nDone. Send this cell\'s full printed output back -- especially the "Verdict" section.')
